# Archived coursework source
Recovered from the author's Google Drive on 2026-10-27.
This historical code is preserved for provenance, not the supported entry point.
Outputs, images, and Colab metadata were removed. The EMNIST export/download cells were omitted.
For EMNIST, the original loop used the official test split for model selection and did not correct image orientation.
Use ../Handwritten_Text_Recognition.ipynb or the htr package for the revised workflow.


In [ ]:
# ===========================================================
# Ячейка 1: Установка и импорт библиотек
# ===========================================================

# Установка дополнительных библиотек (в Colab PyTorch уже есть)
# !pip install torchvision matplotlib numpy pillow

import torch                          # Основной фреймворк глубокого обучения
import torch.nn as nn                 # Модуль для построения нейронных сетей
import torch.optim as optim           # Оптимизаторы (Adam, SGD и др.)
import torch.nn.functional as F       # Функции активации, потерь и т.д.

from torch.utils.data import DataLoader  # Загрузчик данных батчами

import torchvision                    # Утилиты для работы с изображениями
import torchvision.transforms as transforms  # Преобразования изображений
from torchvision.datasets import EMNIST      # Датасет EMNIST (буквы + цифры)

import matplotlib.pyplot as plt       # Построение графиков
import matplotlib.patches as mpatches
import numpy as np                    # Работа с массивами

from PIL import Image                 # Открытие пользовательских изображений
import os                             # Работа с файловой системой
import shutil                         # Копирование/архивация файлов
import zipfile                        # Создание .zip архива
import time                           # Замер времени обучения

# Проверяем, доступен ли GPU (в Colab включите Runtime → Change runtime type → T4 GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Используемое устройство: {device}")
if device.type == "cuda":
    print(f"[INFO] GPU: {torch.cuda.get_device_name(0)}")
    print(f"[INFO] Память GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ===========================================================
# Ячейка 2: Гиперпараметры и конфигурация
# ===========================================================

# --- Гиперпараметры обучения ---
BATCH_SIZE    = 128    # Количество примеров в одном батче (чем больше — тем быстрее, но нужно больше RAM)
LEARNING_RATE = 0.001  # Шаг обучения для оптимизатора Adam
NUM_EPOCHS    = 15     # Количество эпох (полных проходов по датасету)
DROPOUT_RATE  = 0.5    # Вероятность "выключения" нейрона во время обучения (регуляризация)
NUM_CLASSES   = 47     # В EMNIST balanced: 10 цифр + 26 прописных + 11 строчных букв = 47 классов
IMG_SIZE      = 28     # Размер входного изображения 28×28 пикселей

# --- Пути для сохранения ---
MODEL_SAVE_PATH = "/content/model_best.pth"  # Путь для сохранения лучшей модели

# Маппинг: индекс класса → символ (EMNIST balanced)
# 0-9: цифры, 10-35: A-Z, 36-46: a-e, g, h, n, q, r, t
EMNIST_LABELS = (
    [str(i) for i in range(10)] +         # '0' – '9'
    [chr(ord('A') + i) for i in range(26)] +  # 'A' – 'Z'
    ['a', 'b', 'd', 'e', 'f', 'g', 'h', 'n', 'q', 'r', 't']  # строчные без двойников
)

print(f"[INFO] Количество классов: {NUM_CLASSES}")
print(f"[INFO] Метки классов: {EMNIST_LABELS}")

In [ ]:
# ===========================================================
# Ячейка 3: Загрузка и предобработка датасета EMNIST
# ===========================================================

# Определяем преобразования для обучающей выборки (аугментация + нормализация)
train_transform = transforms.Compose([
    transforms.RandomRotation(10),          # Случайный поворот ±10° (имитирует реальный почерк)
    transforms.RandomAffine(0, translate=(0.1, 0.1)),  # Небольшой случайный сдвиг
    transforms.ToTensor(),                  # Конвертация PIL Image → Tensor [0,1]
    transforms.Normalize((0.1307,), (0.3081,))  # Нормализация: среднее и ст. отклонение MNIST
])

# Преобразования для тестовой выборки (только нормализация, без аугментации)
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

print("[INFO] Загрузка датасета EMNIST balanced (это займёт ~1-2 минуты)...")

# Загрузка обучающего набора
# split="balanced" — сбалансированная версия с 47 классами
train_dataset = EMNIST(
    root="/content/data",      # Папка для скачивания данных
    split="balanced",           # Сбалансированная версия (47 классов)
    train=True,                 # Обучающая часть
    download=True,              # Скачать если не скачан
    transform=train_transform   # Применяемые преобразования
)

# Загрузка тестового набора
test_dataset = EMNIST(
    root="/content/data",
    split="balanced",
    train=False,                # Тестовая часть
    download=True,
    transform=test_transform
)

# DataLoader — обёртка для эффективной подачи данных в модель
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,      # Размер батча
    shuffle=True,               # Перемешивать данные каждую эпоху
    num_workers=2,              # Параллельная загрузка данных (2 потока)
    pin_memory=True             # Ускорение передачи данных на GPU
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,              # Тест не перемешиваем
    num_workers=2,
    pin_memory=True
)

print(f"[INFO] Обучающих примеров: {len(train_dataset):,}")
print(f"[INFO] Тестовых примеров:  {len(test_dataset):,}")
print(f"[INFO] Батчей в эпохе:     {len(train_loader)}")

# Визуализация нескольких примеров из датасета
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
fig.suptitle("Примеры из датасета EMNIST balanced", fontsize=14, fontweight='bold')

for i, ax in enumerate(axes.flat):
    img, label = train_dataset[i]
    # img.squeeze() убирает размерность канала (1, 28, 28) → (28, 28)
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(EMNIST_LABELS[label], fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.savefig("/content/dataset_examples.png", dpi=120, bbox_inches='tight')
plt.show()
print("[INFO] Примеры датасета сохранены в /content/dataset_examples.png")

In [ ]:
# ===========================================================
# Ячейка 4: Архитектура CNN (Сверточная нейронная сеть)
# ===========================================================

class HandwritingCNN(nn.Module):
    """
    Сверточная нейронная сеть для распознавания рукописных символов.

    Архитектура:
    Блок 1: Conv(1→32) → BN → ReLU → Conv(32→32) → BN → ReLU → MaxPool → Dropout(0.25)
    Блок 2: Conv(32→64) → BN → ReLU → Conv(64→64) → BN → ReLU → MaxPool → Dropout(0.25)
    Блок 3: Conv(64→128) → BN → ReLU → MaxPool → Dropout(0.25)
    FC:     Flatten → Linear(128→256) → ReLU → Dropout(0.5) → Linear(256→NUM_CLASSES)
    """

    def __init__(self, num_classes=NUM_CLASSES):
        super(HandwritingCNN, self).__init__()

        # === СВЕРТОЧНЫЙ БЛОК 1 ===
        # Входной канал: 1 (grayscale), выход: 32 карты признаков, ядро 3×3
        self.conv1_1 = nn.Conv2d(
            in_channels=1,     # 1 канал (ч/б изображение)
            out_channels=32,   # 32 фильтра — каждый ищет свой паттерн
            kernel_size=3,     # Ядро свёртки 3×3
            padding=1          # Нулевое заполнение, чтобы сохранить размер 28×28
        )
        self.bn1_1 = nn.BatchNorm2d(32)  # Батч-нормализация стабилизирует обучение

        self.conv1_2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.bn1_2 = nn.BatchNorm2d(32)

        # MaxPool2d(2,2): уменьшает размер с 28×28 до 14×14
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.drop1 = nn.Dropout2d(p=0.25)  # Dropout для сверточных слоёв

        # === СВЕРТОЧНЫЙ БЛОК 2 ===
        # Входной канал: 32 (от блока 1), выход: 64 карты признаков
        self.conv2_1 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2_1 = nn.BatchNorm2d(64)

        self.conv2_2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn2_2 = nn.BatchNorm2d(64)

        # MaxPool2d: уменьшает с 14×14 до 7×7
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.drop2 = nn.Dropout2d(p=0.25)

        # === СВЕРТОЧНЫЙ БЛОК 3 ===
        self.conv3_1 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3_1 = nn.BatchNorm2d(128)

        # MaxPool2d: уменьшает с 7×7 до 3×3
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.drop3 = nn.Dropout2d(p=0.25)

        # === ПОЛНОСВЯЗНЫЕ СЛОИ (Classifier) ===
        # После pool3: тензор 128 × 3 × 3 = 1152 значения
        self.fc1 = nn.Linear(128 * 3 * 3, 256)   # 1152 → 256
        self.bn_fc = nn.BatchNorm1d(256)            # Нормализация для FC слоя
        self.drop_fc = nn.Dropout(p=DROPOUT_RATE)   # Dropout 50%
        self.fc2 = nn.Linear(256, num_classes)       # 256 → 47 классов

        # Инициализация весов метод Kaiming (рекомендован для ReLU)
        self._initialize_weights()

    def _initialize_weights(self):
        """Инициализация весов по методу He (Kaiming)"""
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        """
        Прямой проход через сеть.
        x: входной тензор [batch, 1, 28, 28]
        returns: логиты [batch, num_classes]
        """
        # --- Блок 1 ---
        x = F.relu(self.bn1_1(self.conv1_1(x)))   # [B, 32, 28, 28]
        x = F.relu(self.bn1_2(self.conv1_2(x)))   # [B, 32, 28, 28]
        x = self.drop1(self.pool1(x))              # [B, 32, 14, 14]

        # --- Блок 2 ---
        x = F.relu(self.bn2_1(self.conv2_1(x)))   # [B, 64, 14, 14]
        x = F.relu(self.bn2_2(self.conv2_2(x)))   # [B, 64, 14, 14]
        x = self.drop2(self.pool2(x))              # [B, 64, 7, 7]

        # --- Блок 3 ---
        x = F.relu(self.bn3_1(self.conv3_1(x)))   # [B, 128, 7, 7]
        x = self.drop3(self.pool3(x))              # [B, 128, 3, 3]

        # --- Переход к полносвязным слоям ---
        x = x.view(x.size(0), -1)                 # Flatten: [B, 1152]

        # --- Классификатор ---
        x = F.relu(self.bn_fc(self.fc1(x)))        # [B, 256]
        x = self.drop_fc(x)                        # Dropout
        x = self.fc2(x)                            # [B, 47] — логиты

        return x


# Создаём модель и переносим на GPU (если доступен)
model = HandwritingCNN(num_classes=NUM_CLASSES).to(device)

# Выводим структуру модели
print(model)
print(f"\n[INFO] Количество параметров: {sum(p.numel() for p in model.parameters()):,}")
print(f"[INFO] Обучаемых параметров:  {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# ===========================================================
# Ячейка 5: Функции обучения и валидации
# ===========================================================

# Функция потерь: Cross-Entropy Loss (стандарт для многоклассовой классификации)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)  # label_smoothing немного улучшает обобщение

# Оптимизатор Adam: адаптивный момент оценки, обычно лучше SGD для задач CV
optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4  # L2 регуляризация — штраф за большие веса
)

# Планировщик скорости обучения: уменьшает LR когда метрика перестаёт улучшаться
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',       # Следим за минимизацией (val_loss)
    factor=0.5,       # Делить LR на 2
    patience=2        # Подождать 2 эпохи без улучшения
)


def train_epoch(model, loader, criterion, optimizer, device):
    """
    Одна эпоха обучения.
    Возвращает: (средний loss, точность в %)
    """
    model.train()    # Включаем режим обучения (Dropout активен, BatchNorm обновляется)

    total_loss    = 0.0
    correct       = 0
    total_samples = 0

    for batch_idx, (images, labels) in enumerate(loader):
        # Переносим данные на GPU/CPU
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # Обнуляем градиенты из предыдущего батча
        optimizer.zero_grad()

        # Прямой проход: получаем предсказания
        outputs = model(images)              # [batch, num_classes]

        # Вычисляем потери
        loss = criterion(outputs, labels)

        # Обратный проход: вычисляем градиенты
        loss.backward()

        # Обрезаем градиенты для предотвращения взрыва
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # Шаг оптимизатора: обновляем веса
        optimizer.step()

        # Накапливаем статистику
        total_loss    += loss.item() * images.size(0)     # Суммарный loss по батчу
        predicted      = outputs.argmax(dim=1)             # Класс с максимальным логитом
        correct       += (predicted == labels).sum().item()
        total_samples += images.size(0)

        # Вывод прогресса каждые 200 батчей
        if (batch_idx + 1) % 200 == 0:
            batch_acc  = 100.0 * correct / total_samples
            batch_loss = total_loss / total_samples
            print(f"  Батч [{batch_idx+1}/{len(loader)}] | Loss: {batch_loss:.4f} | Acc: {batch_acc:.2f}%")

    avg_loss = total_loss / total_samples
    accuracy = 100.0 * correct / total_samples
    return avg_loss, accuracy


def validate(model, loader, criterion, device):
    """
    Валидация модели на тестовой выборке.
    Возвращает: (средний loss, точность в %)
    """
    model.eval()    # Режим вывода: Dropout выключен, BatchNorm использует скользящие средние

    total_loss    = 0.0
    correct       = 0
    total_samples = 0

    with torch.no_grad():    # Отключаем вычисление градиентов → экономия памяти
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs   = model(images)
            loss      = criterion(outputs, labels)

            total_loss    += loss.item() * images.size(0)
            predicted      = outputs.argmax(dim=1)
            correct       += (predicted == labels).sum().item()
            total_samples += images.size(0)

    avg_loss = total_loss / total_samples
    accuracy = 100.0 * correct / total_samples
    return avg_loss, accuracy

print("[INFO] Функции обучения и валидации определены.")

In [ ]:
# ===========================================================
# Ячейка 6: Цикл обучения (Train Loop)
# ===========================================================

# Словари для хранения истории метрик
history = {
    "train_loss": [],
    "train_acc":  [],
    "val_loss":   [],
    "val_acc":    []
}

best_val_acc  = 0.0      # Лучшая точность на валидации
best_val_loss = float('inf')

print("=" * 65)
print(f"{'НАЧАЛО ОБУЧЕНИЯ':^65}")
print("=" * 65)
print(f"Эпох: {NUM_EPOCHS} | Батч: {BATCH_SIZE} | LR: {LEARNING_RATE} | Device: {device}")
print("=" * 65)

total_start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start = time.time()

    print(f"\n[Эпоха {epoch:02d}/{NUM_EPOCHS}]")

    # --- Обучение ---
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)

    # --- Валидация ---
    val_loss, val_acc = validate(model, test_loader, criterion, device)

    # --- Обновление планировщика LR ---
    scheduler.step(val_loss)

    # --- Запись метрик ---
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    epoch_time = time.time() - epoch_start

    # --- Вывод результатов эпохи ---
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"  Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.2f}%")
    print(f"  Время эпохи: {epoch_time:.1f} сек.")

    # --- Сохранение лучшей модели ---
    if val_acc > best_val_acc:
        best_val_acc  = val_acc
        best_val_loss = val_loss
        torch.save({
            "epoch":       epoch,
            "model_state": model.state_dict(),
            "optimizer":   optimizer.state_dict(),
            "val_acc":     val_acc,
            "val_loss":    val_loss,
            "num_classes": NUM_CLASSES,
            "labels":      EMNIST_LABELS
        }, MODEL_SAVE_PATH)
        print(f"  ✅ Лучшая модель сохранена! Val Acc: {best_val_acc:.2f}%")

total_time = time.time() - total_start
print("\n" + "=" * 65)
print(f"ОБУЧЕНИЕ ЗАВЕРШЕНО за {total_time/60:.1f} минут")
print(f"Лучшая точность на тесте: {best_val_acc:.2f}%")
print("=" * 65)

In [ ]:
# ===========================================================
# Ячейка 7: Визуализация графиков обучения
# ===========================================================

epochs_range = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Процесс обучения CNN на EMNIST", fontsize=15, fontweight='bold')

# --- График функции потерь ---
ax1 = axes[0]
ax1.plot(epochs_range, history["train_loss"], 'b-o', markersize=4, label='Обучение (Train Loss)', linewidth=2)
ax1.plot(epochs_range, history["val_loss"],   'r-o', markersize=4, label='Валидация (Val Loss)',  linewidth=2)
ax1.set_xlabel("Эпоха", fontsize=12)
ax1.set_ylabel("Cross-Entropy Loss", fontsize=12)
ax1.set_title("Функция потерь", fontsize=13)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(1, len(epochs_range))

# --- График точности ---
ax2 = axes[1]
ax2.plot(epochs_range, history["train_acc"], 'b-o', markersize=4, label='Обучение (Train Acc)', linewidth=2)
ax2.plot(epochs_range, history["val_acc"],   'r-o', markersize=4, label='Валидация (Val Acc)',  linewidth=2)
ax2.axhline(y=best_val_acc, color='green', linestyle='--', alpha=0.7, label=f'Лучшая: {best_val_acc:.2f}%')
ax2.set_xlabel("Эпоха", fontsize=12)
ax2.set_ylabel("Точность (%)", fontsize=12)
ax2.set_title("Точность классификации", fontsize=13)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.set_xlim(1, len(epochs_range))
ax2.set_ylim(0, 100)

plt.tight_layout()
plt.savefig("/content/training_curves.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"[INFO] Графики сохранены в /content/training_curves.png")

In [ ]:
# ===========================================================
# Ячейка 8: Функция инференса (предсказание для одного изображения)
# ===========================================================

def load_model(path, device):
    """Загружает сохранённую модель из файла .pth"""
    checkpoint = torch.load(path, map_location=device)
    model = HandwritingCNN(num_classes=checkpoint["num_classes"]).to(device)
    model.load_state_dict(checkpoint["model_state"])
    model.eval()
    labels = checkpoint.get("labels", EMNIST_LABELS)
    print(f"[INFO] Модель загружена. Точность на тесте при сохранении: {checkpoint['val_acc']:.2f}%")
    return model, labels


def preprocess_image(image_path):
    """
    Подготавливает изображение для подачи в модель.
    Принимает: путь к изображению (JPG, PNG, BMP)
    Возвращает: тензор [1, 1, 28, 28]
    """
    img = Image.open(image_path).convert("L")    # Конвертируем в оттенки серого
    img = img.resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS)  # Меняем размер на 28×28

    img_array = np.array(img, dtype=np.float32)  # В numpy массив

    # Инверсия: EMNIST обучен на белом тексте на чёрном фоне
    # Если ваш рисунок — чёрный текст на белом фоне, инвертируем
    if img_array.mean() > 128:   # Если фон светлый → инвертируем
        img_array = 255.0 - img_array

    img_tensor = torch.tensor(img_array / 255.0, dtype=torch.float32)  # Нормализуем [0,1]
    img_tensor = (img_tensor - 0.1307) / 0.3081                          # Стандартизация
    img_tensor = img_tensor.unsqueeze(0).unsqueeze(0)                     # [1, 1, 28, 28]

    return img_tensor, img_array


def predict(model, image_path, labels, device, top_k=5):
    """
    Распознаёт символ на изображении.
    Возвращает топ-K предсказаний с вероятностями.
    """
    img_tensor, img_array = preprocess_image(image_path)
    img_tensor = img_tensor.to(device)

    model.eval()
    with torch.no_grad():
        logits = model(img_tensor)                     # Логиты [1, 47]
        probs  = F.softmax(logits, dim=1)              # Вероятности через Softmax

        # Топ-K классов
        top_probs, top_indices = torch.topk(probs, k=top_k, dim=1)
        top_probs   = top_probs.squeeze().cpu().numpy()
        top_indices = top_indices.squeeze().cpu().numpy()

    # --- Визуализация результата ---
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    fig.suptitle("Результат распознавания", fontsize=14, fontweight='bold')

    # Исходное изображение
    axes[0].imshow(img_array, cmap='gray')
    axes[0].set_title(f"Входное изображение\nПредсказание: «{labels[top_indices[0]]}»", fontsize=12)
    axes[0].axis('off')

    # График вероятностей
    top_labels = [labels[i] for i in top_indices]
    colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(top_k)]
    bars = axes[1].barh(range(top_k), top_probs * 100, color=colors, edgecolor='white')
    axes[1].set_yticks(range(top_k))
    axes[1].set_yticklabels([f"«{l}»" for l in top_labels], fontsize=12)
    axes[1].set_xlabel("Вероятность (%)", fontsize=11)
    axes[1].set_title(f"Топ-{top_k} предсказаний", fontsize=12)
    axes[1].set_xlim(0, 100)

    # Подписи на барах
    for i, (bar, prob) in enumerate(zip(bars, top_probs)):
        axes[1].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                     f"{prob*100:.1f}%", va='center', fontsize=10)

    plt.tight_layout()
    plt.savefig("/content/prediction_result.png", dpi=150, bbox_inches='tight')
    plt.show()

    print(f"\n{'='*45}")
    print(f"  Распознанный символ:   «{labels[top_indices[0]]}»")
    print(f"  Уверенность модели:     {top_probs[0]*100:.2f}%")
    print(f"  Топ-{top_k} кандидатов:")
    for i, (lbl, prob) in enumerate(zip(top_labels, top_probs)):
        bar = '█' * int(prob * 30)
        print(f"    {i+1}. «{lbl}»  {prob*100:6.2f}%  {bar}")
    print(f"{'='*45}")

    return labels[top_indices[0]], top_probs[0]

In [ ]:
# ===========================================================
# Ячейка 9: Демонстрация инференса на тестовых примерах
# ===========================================================

# Загружаем лучшую модель
model_inf, labels_inf = load_model(MODEL_SAVE_PATH, device)

# --- Тест 1: Берём случайные примеры из тестового набора ---
print("\n[TEST] Предсказания на случайных примерах из тестового набора:")

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
fig.suptitle("Предсказания модели на тестовом наборе", fontsize=14)

correct_count = 0
sample_indices = np.random.randint(0, len(test_dataset), size=10)

for ax, idx in zip(axes.flat, sample_indices):
    img_tensor, true_label = test_dataset[idx]

    img_input = img_tensor.unsqueeze(0).to(device)   # [1, 1, 28, 28]
    with torch.no_grad():
        logits    = model_inf(img_input)
        pred_idx  = logits.argmax(dim=1).item()
        confidence = F.softmax(logits, dim=1).max().item()

    pred_label = labels_inf[pred_idx]
    true_char  = labels_inf[true_label]
    is_correct = (pred_idx == true_label)
    correct_count += int(is_correct)

    ax.imshow(img_tensor.squeeze(), cmap='gray')
    color = 'green' if is_correct else 'red'
    ax.set_title(f"Истина: {true_char}\nПред: {pred_label} ({confidence*100:.0f}%)",
                 color=color, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig("/content/batch_predictions.png", dpi=120, bbox_inches='tight')
plt.show()
print(f"[TEST] Правильно из 10 случайных: {correct_count}/10")

In [ ]:
# ===========================================================
# Ячейка 10: Загрузка собственного изображения (для Colab)
# ===========================================================

# ИНСТРУКЦИЯ:
# 1. Нарисуйте букву или цифру от руки, сфотографируйте
# 2. Загрузите файл в Colab через левую панель (значок папки → загрузить)
# 3. Укажите путь к файлу в USER_IMAGE_PATH ниже

USER_IMAGE_PATH = "/content/X.png"   # ← Укажите ваш путь

# Если файл существует — распознаём его
if os.path.exists(USER_IMAGE_PATH):
    print(f"[INFO] Распознаём изображение: {USER_IMAGE_PATH}")
    predicted_char, confidence = predict(model_inf, USER_IMAGE_PATH, labels_inf, device)
else:
    # Создаём демо-изображение для примера (цифра '5')
    print(f"[INFO] Файл {USER_IMAGE_PATH} не найден. Создаём синтетический пример...")

    # Берём реальный пример из тестового набора с меткой '5'
    for idx in range(len(test_dataset)):
        img_t, lbl = test_dataset[idx]
        if labels_inf[lbl] == '5':
            demo_img = (img_t.squeeze().numpy() * 0.3081 + 0.1307)
            demo_img = np.clip(demo_img * 255, 0, 255).astype(np.uint8)
            Image.fromarray(demo_img).save(USER_IMAGE_PATH)
            print(f"[INFO] Сохранено демо-изображение цифры '5' → {USER_IMAGE_PATH}")
            break

    predicted_char, confidence = predict(model_inf, USER_IMAGE_PATH, labels_inf, device)